In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
! pip install sentence-transformers

In [4]:
import pandas as pd

In [5]:
train_df = pd.read_csv('/content/drive/MyDrive/fake-news/train_clean.csv')
test_df = pd.read_csv('/content/drive/MyDrive/fake-news/test_clean.csv')
val_df = pd.read_csv('/content/drive/MyDrive/fake-news/val_clean.csv')

In [6]:
print(train_df.shape)
print(train_df.head())

(10240, 2)
                                           statement      label
0  says the annies list political group supports ...       fake
1  when did the decline of coal start? it started...  uncertain
2  hillary clinton agrees with john mccain "by vo...       real
3  health care reform legislation is likely to ma...       fake
4  the economic turnaround started at the end of ...  uncertain


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!


In [9]:
sample = train_df['statement'].head (1000)

In [10]:
print(sample.shape)

(1000,)


In [11]:
embeddings = model.encode(sample.tolist(),show_progress_bar=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [13]:
print(embeddings.shape)

(1000, 384)


In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# pick the first claim as query
query_embedding = embeddings[0].reshape(1, -1)
query_text = sample.iloc[0]

# compute similarity against all 1000
similarities = cosine_similarity(query_embedding, embeddings)[0]

# get top 5 most similar (excluding itself)
top_indices = np.argsort(similarities)[::-1][1:6]

print(f"Query: {query_text}\n")
print("Top 5 similar claims:")
for idx in top_indices:
    print(f"  [{similarities[idx]:.2f}] {sample.iloc[idx]}")

Query: says the annies list political group supports third-trimester abortions on demand.

Top 5 similar claims:
  [0.57] hillary clinton supports unlimited abortion on demand up until the moment of birth, including partial-birth abortion, with taxpayer funding.
  [0.55] on abortion
  [0.52] oregonians have an amazing no-cost way to fight abortion with free political donations
  [0.51] says cory booker supports late-term and partial-birth abortion and opposes safety regulations.
  [0.50] says newt gingrich made an affirmative statement that he would not only support but he would campaign for republicans who were in support of the barbaric procedure known as partial-birth abortion.


In [16]:
# find the least similar claim
bottom_indices = np.argsort(similarities)[0:3]

print("3 LEAST similar claims to the query:")
for idx in bottom_indices:
    print(f"  [{similarities[idx]:.2f}] {sample.iloc[idx]}")

3 LEAST similar claims to the query:
  [-0.12] if the stamp farm wind turbine is built, the health risk of flicker impact created by shadows of blades of turbines poses real and significant health risks, particularly seizures.
  [-0.09] the difference over the (previous) four years between where things were headed and where we brought them to over the last four years is a total of $816 in saved property tax money.
  [-0.08] on how money from a possible lease of the ohio turnpike would be used
